# RMSX: Interactive Protein Motion with Flipbook

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AntunesLab/rmsx/blob/main/RMSX_Molstar_Colab_Demo.ipynb)

RMSX maps **where and when a protein moves** during a molecular dynamics trajectory. It divides the trajectory into time slices, measures residue-level structural variation, and summarizes the result as plots and representative structures.

Flipbook turns those results into an interactive molecular view. Each structure represents one time slice, while color and backbone thickness show the RMSX value at each residue. You can rotate, zoom, and adjust the layout directly in this notebook.

We'll cover:

1. Set up RMSX in Google Colab
2. Analyze a single-chain protein
3. Analyze a multi-chain protein
4. Explore an interactive Flipbook
5. Mask selected residues when comparing motion

> In Colab, choose **Runtime > Run all**. The first setup cells may take a few minutes because RMSX, MDAnalysis, and R are installed into the temporary Colab runtime. RMSX installs its R plotting packages automatically the first time it creates a plot.


## 1) Environment Setup

RMSX uses Python/MDAnalysis for trajectory analysis, R for heatmaps and RMSD/RMSF plots, and Molstar for the inline 3D viewer. The complete Python install is one command from PyPI.


In [ ]:
%pip install -q --upgrade rmsx
!rmsx --version


### R Plotting Packages

RMSX uses R to generate its heatmap, RMSD, and RMSF plots. PyPI cannot install R itself, so this is the only separate system dependency for the complete demo. RMSX installs its required R packages automatically when it generates the first plot.


In [ ]:
!apt-get update -qq && apt-get install -y -qq r-base


## 2) Load Demo Input Files

The notebook looks for the bundled demo inputs inside the installed `rmsx` package first. If you are running from a source checkout, it also checks local `test_files` folders. Demo outputs are written to `./rmsx_demo_outputs` so the packaged inputs remain read-only.


In [ ]:
from pathlib import Path
import os
import subprocess

import rmsx
from rmsx import all_chain_rmsx, run_rmsx, run_rmsx_flipbook

pkg_dir = Path(rmsx.__file__).resolve().parent
env_override = os.environ.get("RMSX_TEST_DIR")

candidates = []
if env_override:
    candidates.append(Path(env_override))

candidates += [
    pkg_dir / "test_files",
    Path.cwd() / "test_files",
    Path.cwd() / "rmsx" / "test_files",
    pkg_dir.parent / "test_files",
]

test_dir = next((p for p in candidates if p.exists()), None)

if not test_dir:
    repo_dir = Path.cwd() / "rmsx"
    if not repo_dir.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/AntunesLab/rmsx.git", str(repo_dir)])
    test_dir = repo_dir / "rmsx" / "test_files"
    if not test_dir.exists():
        test_dir = repo_dir / "test_files"
    if not test_dir.exists():
        raise FileNotFoundError("Could not locate bundled RMSX demo inputs.")

demo_output_root = Path.cwd() / "rmsx_demo_outputs"
demo_output_root.mkdir(exist_ok=True)

pdb_file = (test_dir / "1UBQ.pdb").as_posix()
dcd_file = (test_dir / "mon_sys.dcd").as_posix()
output_dir = (demo_output_root / "example_uqb").as_posix()

pdb_file_multi = (test_dir / "protease_backbone.pdb").as_posix()
traj_file_multi = (test_dir / "short_protease_backbone.dcd").as_posix()
output_dir_multi = (demo_output_root / "protease").as_posix()

print("RMSX package:", Path(rmsx.__file__).resolve())
print("Demo input directory:", test_dir)
print("Demo output root:", demo_output_root.resolve())
print("Single-chain input:", pdb_file, dcd_file, sep="\n  ")
print("Multi-chain input:", pdb_file_multi, traj_file_multi, sep="\n  ")


## 3) Single-Chain RMSX

Start with the bundled Ubiquitin trajectory. `run_rmsx` computes residue-level RMSX for each time slice and writes the RMSX heatmap, RMSD/RMSF plots, data tables, and representative PDB structures. The chain ID in this example is `"7"`.


In [ ]:
run_rmsx(
    topology_file=pdb_file,
    trajectory_file=dcd_file,
    output_dir=output_dir,
    num_slices=9,
    slice_size=None,
    rscript_executable=os.environ.get("RSCRIPT", "Rscript"),
    verbose=False,
    interpolate=False,
    triple=True,
    overwrite=True,
    palette="mako",
    chain_sele="7",
    start_frame=0,
    end_frame=None,
    full_backbone=True,
)

print("Single-chain RMSX output:", output_dir)


## 4) Multi-Chain RMSX

The protease demo runs RMSX on each chain and synchronizes the color scale so chain-level plots and the combined Flipbook use a consistent RMSX range.


In [ ]:
all_chain_rmsx(
    topology_file=pdb_file_multi,
    trajectory_file=traj_file_multi,
    output_dir=output_dir_multi,
    num_slices=9,
    slice_size=None,
    rscript_executable=os.environ.get("RSCRIPT", "Rscript"),
    verbose=False,
    interpolate=False,
    triple=True,
    overwrite=True,
    palette="turbo",
    start_frame=0,
    end_frame=None,
    sync_color_scale=True,
    full_backbone=True,
)


## 5) Interactive Molstar Flipbook

`run_rmsx_flipbook` combines the RMSX analysis with an interactive view of the representative structures. Molstar runs in the browser, so the Flipbook appears directly below the cell. `spacingFactor=0.42` starts the nine slices in a compact, single-row layout; adjust that one value if your protein needs more or less room.


In [ ]:
run_rmsx_flipbook(
    topology_file=pdb_file_multi,
    trajectory_file=traj_file_multi,
    output_dir=output_dir_multi,
    num_slices=9,
    slice_size=None,
    rscript_executable=os.environ.get("RSCRIPT", "Rscript"),
    verbose=False,
    interpolate=False,
    triple=True,
    overwrite=True,
    palette="turbo",
    spacingFactor=0.42,  # Compact spacing; increase if structures overlap.
    viewer="molstar",
    molstar_asset_mode="cdn",
    start_frame=0,
    end_frame=None,
)


## 6) Masked Molstar Flipbook

Masking is useful when a very mobile or disordered region dominates the RMSX scale. Masked residues are excluded from the range estimate, shown with hatch overlays in heatmaps, and rendered semi-transparent in Molstar.


In [ ]:
protease_active_site_mask = [
    "segid A and resid 45:55",
    "segid B and resid 45:55",
]

output_dir_mask_demo = demo_output_root / "protease_mask_example"

run_rmsx_flipbook(
    topology_file=pdb_file_multi,
    trajectory_file=traj_file_multi,
    output_dir=output_dir_mask_demo,
    num_slices=9,
    rscript_executable=os.environ.get("RSCRIPT", "Rscript"),
    verbose=False,
    interpolate=False,
    triple=True,
    overwrite=True,
    palette="magma",
    sync_color_scale=True,
    mask=protease_active_site_mask,
    spacingFactor=0.42,  # Compact spacing; increase if structures overlap.
    viewer="molstar",
    molstar_asset_mode="cdn",
)


## Use Your Own Data

Replace `topology_file`, `trajectory_file`, and `output_dir` with your own files. For Colab, upload files through the left sidebar or mount Google Drive first.

```python
from rmsx import run_rmsx_flipbook

run_rmsx_flipbook(
    topology_file="/content/my_structure.pdb",
    trajectory_file="/content/my_trajectory.dcd",
    output_dir="/content/my_rmsx_flipbook",
    num_slices=9,
    chain_sele=None,
    palette="turbo",
    viewer="molstar",
    molstar_asset_mode="cdn",
    overwrite=True,
)
```

### Running Locally

This notebook uses Molstar because it runs directly in a web browser. When running RMSX locally, you can also use ChimeraX or VMD by setting `viewer="chimerax"` or `viewer="vmd"`. Those desktop applications must be installed separately.


## Citation

If you use RMSX + Flipbook in your work, please cite:

Beruldsen, F., de Freitas, M.V. & Antunes, D.A. *High resolution mapping of protein motions in time and space with RMSX and Flipbook.* **Scientific Reports** (2026). https://doi.org/10.1038/s41598-026-39869-7
